In [22]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [32]:
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a movie summarizer'),
    ('human', 'Please summarize the movie in brief : {input}')
])

In [24]:
#TASK -2

llm = init_chat_model(model="ollama:gemma3:4b")

In [25]:
#TASK - 3 :
str_parser = StrOutputParser()

## Parallel chain 1

In [26]:
# TASK  - 4 :
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text : str) -> dict:
    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

In [27]:
#TASK -1 # create prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Linkedin Post Generator'),
    ('human', 'Create a post for the following text for Linkedin : {text}')
])

#TASK -2

llm = init_chat_model(model="ollama:gemma3:4b")

#TASK - 3 :
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser


## Parallel chain 2

In [28]:
def insta_chain(text : dict):

    text = text["text"]

    insta_prompt = ChatPromptTemplate.from_messages([
        ('system','You are a instagram post generator'),
        ('human', 'Create a post for the following text for Instagram : {text}')
    ])

    llm = init_chat_model(model="ollama:gemma3:4b")

    str_parser = StrOutputParser()

    insta_chain = insta_prompt | llm | str_parser

    result  = insta_chain.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)


## Final orchestration

In [29]:
from langchain_core.runnables import RunnableParallel

In [34]:
final_chain = (
                prompt_template |
                llm |
                str_parser |
                dictionary_maker_runnable |
                RunnableParallel(branches={'linkedin' : chain_linkedin, 'instagram' : insta_chain_runnable})
)

In [35]:
final_chain.invoke("KGF")

{'branches': {'linkedin': "Okay, here are a few LinkedIn post options based on your text, catering to different tones and lengths. Choose the one that best suits your style and audience:\n\n**Option 1 (Concise & Engaging - Best for Quick Engagement)**\n\n🔥 Just finished diving into the explosive world of K.G.F: Chapter 1! 🔥 This action epic about Rocky’s relentless quest for revenge is a must-watch. Driven by ambition and fueled by betrayal, he transforms from a humble orphan to a legendary mercenary. 🤯  #KGF #Rocky #ActionMovie #Revenge #Bollywood #MustWatch\n\n**Option 2 (Slightly More Detailed - Good for sparking discussion)**\n\nIf you haven't experienced the incredible ride that is K.G.F: Chapter 1, you’re missing out! This high-octane story follows Rocky, an orphan who becomes a ruthless mercenary, driven by a burning desire to dismantle a powerful empire. It's a masterclass in ambition, betrayal, and a truly relentless quest for vengeance.  What were your biggest takeaways from 

## Chain as RUnnable

In [37]:
# TASK [beautify dictinary]

def beautify_final_dict(final_response : dict ) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    insta_response = final_response['branches']['instagram']

    return {'linkedin' : linkedin_response, 'instagram' : insta_response}

beautify_final_dict_runnable  = RunnableLambda(beautify_final_dict)


# TASK 2 final chain

beautify_chain = final_chain | beautify_final_dict_runnable
beautify_chain

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a movie summarizer'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='Please summarize the movie in brief : {input}'), additional_kwargs={})])
| ChatOllama(output_version=None, model='gemma3:4b')
| StrOutputParser()
| RunnableLambda(dictionary_maker)
| {
    branches: {
                linkedin: ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a Linkedin Post Generator'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, 

In [38]:
beautify_chain.invoke("pushpa")

{'linkedin': "Okay, here are a few LinkedIn post options based on that text, ranging in tone and length. Choose the one that best suits your style and audience!\n\n**Option 1 (Concise & Engaging - Best for quick engagement):**\n\n**(Image: A striking still from Pushpa: The Rise - Part 1 featuring Allu Arjun)**\n\n“Absolutely captivated by *Pushpa: The Rise*! 🤩 This epic story follows Pushpa’s journey from a hopeful farmer seeking opportunity to a tragically entangled figure within a ruthless red sandalwood mafia. It’s a powerful tale of survival, corruption, and the dangerous allure of loyalty. 🔥 Have you seen it? #PushpaTheRise #AlluArjun #IndianCinema #Bollywood #MustWatch”\n\n**Option 2 (Slightly More Detailed - Good for sparking discussion):**\n\n**(Image: A dynamic shot of Pushpa and Bheema Shankar)**\n\n“Just finished watching *Pushpa: The Rise* and I'm still thinking about it! This film is a gripping exploration of a naive farmer, Pushpa Raj Reddy, who gets drawn into the danger